In [2]:
# Cell 1: Fine-tune MiniLM on Fault Descriptions and extract fine-tuned embeddings

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AdamW
from tqdm import tqdm

# Load data
# df = pd.read_excel("fault-logs-o-ran-12k.xlsx", engine="openpyxl")
df = pd.read_excel("fault-logs-o-ran-64.xls")

fault_texts = df["Fault Description"].fillna("").astype(str).tolist()


# Load MiniLM tokenizer and model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Custom dataset
class FaultDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = tokenizer(self.texts[idx], padding="max_length", truncation=True, max_length=64, return_tensors="pt")
        return {key: val.squeeze(0) for key, val in encoded.items()}

# DataLoader
dataset = FaultDataset(fault_texts)
loader = DataLoader(dataset, batch_size=16, shuffle=False)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Fine-tune MiniLM (light fine-tuning)
model.train()
for epoch in range(1):  # Keep it low to avoid overfitting
    for batch in tqdm(loader, desc="Fine-tuning MiniLM"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        sentence_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS] token

        # Self-supervised dummy objective (contrastive not implemented here for simplicity)
        loss = sentence_embeddings.norm(p=2, dim=1).mean()  # L2 regularization proxy
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

# Re-extract embeddings using fine-tuned MiniLM
model.eval()
all_embeddings = []

with torch.no_grad():
    for batch in tqdm(loader, desc="Extracting Fine-tuned Embeddings"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(embeddings.cpu())

final_embeddings = torch.cat(all_embeddings, dim=0)
print(f"Final Embedding Shape: {final_embeddings.shape}")  # Shape: [num_samples, hidden_dim]


/home/iiitb/rag-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/iiitb/rag-env/lib/python3.11/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Extracting Fine-tuned Embeddings: 100%|███████████| 4/4 [00:00<00:00, 54.99it/s]

Final Embedding Shape: torch.Size([64, 384])


In [19]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Number of generators
num_generators = 30

# Generator
class Generator(nn.Module):
    def __init__(self, latent_dim=100, output_dim=384):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 256),
            nn.ReLU(True),
            nn.Linear(256, 256),
            nn.ReLU(True),
            nn.Linear(256, output_dim),
        )

    def forward(self, z):
        return self.model(z)

# Create generators
generators = [
    Generator(latent_dim=100, output_dim=384).to(device)
    for _ in range(num_generators)
]

# MAD-GAN Discriminator
class MADDiscriminator(nn.Module):
    def __init__(self, input_dim=384, num_generators=30):
        super(MADDiscriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, num_generators + 1),
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        logits = self.net(x)
        return self.softmax(logits)

# Create discriminator
discriminator = MADDiscriminator(
    input_dim=384,
    num_generators=num_generators
).to(device)

In [20]:
import torch.nn.functional as F

# Hyperparameters
latent_dim = 100
hidden_dim = final_embeddings.shape[1]  # 384
batch_size = 64
num_epochs = 500
lr = 1e-4
K = 30  # Number of generators

# Create multiple generators
generators = [Generator(latent_dim, hidden_dim).to(device) for _ in range(K)]
D = MADDiscriminator(input_dim=hidden_dim, num_generators=K).to(device)

# Optimizers
opt_G = [torch.optim.Adam(G.parameters(), lr=lr) for G in generators]
opt_D = torch.optim.Adam(D.parameters(), lr=lr)

# Loss function
ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()

# Dataset and loader
real_dataset = torch.utils.data.TensorDataset(final_embeddings)
real_loader = torch.utils.data.DataLoader(real_dataset, batch_size=batch_size, shuffle=True)

# Training loop
for epoch in range(num_epochs):
    for x_batch, in real_loader:
        x_batch = x_batch.to(device)
        bsz = x_batch.size(0)

        # ============================
        # 1. Train Discriminator
        # ============================

        # Real data → label 0
        real_labels = torch.zeros(bsz, dtype=torch.long).to(device)  # class 0 = real
        real_preds = D(x_batch)  # shape: [bsz, K+1]
        loss_real = ce_loss(real_preds, real_labels)

        # Fake data from all generators → label j ∈ [1..K]
        fake_data = []
        fake_labels = []
        for j, G in enumerate(generators):
            z = torch.randn(bsz, latent_dim).to(device)
            x_fake = G(z).detach()
            fake_preds = D(x_fake)  # shape: [bsz, K+1]
            labels = torch.full((bsz,), j + 1, dtype=torch.long).to(device)  # class j+1 = G_j
            loss_fake = ce_loss(fake_preds, labels)
            fake_data.append((x_fake, labels, loss_fake))

        # Combine all losses
        loss_D = loss_real + sum(lf for _, _, lf in fake_data)

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # ============================
        # 2. Train each Generator
        # ============================

        loss_G_total = 0.0
        loss_MSE_total = 0.0

        for j, G in enumerate(generators):
            z = torch.randn(bsz, latent_dim).to(device)
            x_fake = G(z)
            preds = D(x_fake)
            target_class = torch.zeros(bsz, dtype=torch.long).to(device)  # Wants to be seen as "real" (class 0)

            loss_G = ce_loss(preds, target_class)

            # Optional MSE term (to guide towards data manifold)
            idx = torch.randint(0, final_embeddings.size(0), (bsz,))
            x_real_ref = final_embeddings[idx].to(device)
            loss_MSE = mse_loss(x_fake, x_real_ref)

            total_loss = loss_G + loss_MSE
            opt_G[j].zero_grad()
            total_loss.backward()
            opt_G[j].step()

            loss_G_total += loss_G.item()
            loss_MSE_total += loss_MSE.item()

    print(f"Epoch {epoch+1} | D Loss: {loss_D.item():.4f} | Avg G Loss: {loss_G_total/K:.4f} | Avg MSE: {loss_MSE_total/K:.4f}")

# Save all generators and discriminator
for j, G in enumerate(generators):
    torch.save(G.state_dict(), f"trained_generator_{j+1}.pt")

torch.save(D.state_dict(), "trained_discriminator.pt")




Epoch 1 | D Loss: 106.4529 | Avg G Loss: 3.4320 | Avg MSE: 0.1059
Epoch 2 | D Loss: 106.4514 | Avg G Loss: 3.4319 | Avg MSE: 0.1056
Epoch 3 | D Loss: 106.4502 | Avg G Loss: 3.4319 | Avg MSE: 0.1047
Epoch 4 | D Loss: 106.4489 | Avg G Loss: 3.4318 | Avg MSE: 0.1039
Epoch 5 | D Loss: 106.4474 | Avg G Loss: 3.4318 | Avg MSE: 0.1032
Epoch 6 | D Loss: 106.4462 | Avg G Loss: 3.4317 | Avg MSE: 0.1022
Epoch 7 | D Loss: 106.4450 | Avg G Loss: 3.4317 | Avg MSE: 0.1022
Epoch 8 | D Loss: 106.4438 | Avg G Loss: 3.4316 | Avg MSE: 0.1018
Epoch 9 | D Loss: 106.4425 | Avg G Loss: 3.4315 | Avg MSE: 0.1002
Epoch 10 | D Loss: 106.4411 | Avg G Loss: 3.4315 | Avg MSE: 0.1000
Epoch 11 | D Loss: 106.4401 | Avg G Loss: 3.4314 | Avg MSE: 0.0994
Epoch 12 | D Loss: 106.4388 | Avg G Loss: 3.4313 | Avg MSE: 0.0986
Epoch 13 | D Loss: 106.4374 | Avg G Loss: 3.4312 | Avg MSE: 0.0979
Epoch 14 | D Loss: 106.4361 | Avg G Loss: 3.4311 | Avg MSE: 0.0969
Epoch 15 | D Loss: 106.4347 | Avg G Loss: 3.4310 | Avg MSE: 0.0968
Epoc

Epoch 124 | D Loss: 106.4709 | Avg G Loss: 3.4086 | Avg MSE: 0.0360
Epoch 125 | D Loss: 106.4706 | Avg G Loss: 3.4099 | Avg MSE: 0.0347
Epoch 126 | D Loss: 106.4717 | Avg G Loss: 3.4111 | Avg MSE: 0.0343
Epoch 127 | D Loss: 106.4715 | Avg G Loss: 3.4122 | Avg MSE: 0.0330
Epoch 128 | D Loss: 106.4712 | Avg G Loss: 3.4134 | Avg MSE: 0.0325
Epoch 129 | D Loss: 106.4711 | Avg G Loss: 3.4145 | Avg MSE: 0.0317
Epoch 130 | D Loss: 106.4708 | Avg G Loss: 3.4154 | Avg MSE: 0.0310
Epoch 131 | D Loss: 106.4708 | Avg G Loss: 3.4163 | Avg MSE: 0.0306
Epoch 132 | D Loss: 106.4693 | Avg G Loss: 3.4172 | Avg MSE: 0.0299
Epoch 133 | D Loss: 106.4682 | Avg G Loss: 3.4181 | Avg MSE: 0.0293
Epoch 134 | D Loss: 106.4687 | Avg G Loss: 3.4189 | Avg MSE: 0.0296
Epoch 135 | D Loss: 106.4689 | Avg G Loss: 3.4196 | Avg MSE: 0.0284
Epoch 136 | D Loss: 106.4680 | Avg G Loss: 3.4203 | Avg MSE: 0.0280
Epoch 137 | D Loss: 106.4667 | Avg G Loss: 3.4210 | Avg MSE: 0.0276
Epoch 138 | D Loss: 106.4673 | Avg G Loss: 3.421

Epoch 248 | D Loss: 106.4536 | Avg G Loss: 3.4297 | Avg MSE: 0.0216
Epoch 249 | D Loss: 106.4539 | Avg G Loss: 3.4297 | Avg MSE: 0.0218
Epoch 250 | D Loss: 106.4544 | Avg G Loss: 3.4297 | Avg MSE: 0.0212
Epoch 251 | D Loss: 106.4534 | Avg G Loss: 3.4297 | Avg MSE: 0.0218
Epoch 252 | D Loss: 106.4540 | Avg G Loss: 3.4297 | Avg MSE: 0.0219
Epoch 253 | D Loss: 106.4539 | Avg G Loss: 3.4297 | Avg MSE: 0.0216
Epoch 254 | D Loss: 106.4540 | Avg G Loss: 3.4297 | Avg MSE: 0.0215
Epoch 255 | D Loss: 106.4543 | Avg G Loss: 3.4297 | Avg MSE: 0.0221
Epoch 256 | D Loss: 106.4539 | Avg G Loss: 3.4297 | Avg MSE: 0.0214
Epoch 257 | D Loss: 106.4546 | Avg G Loss: 3.4297 | Avg MSE: 0.0210
Epoch 258 | D Loss: 106.4539 | Avg G Loss: 3.4297 | Avg MSE: 0.0216
Epoch 259 | D Loss: 106.4542 | Avg G Loss: 3.4297 | Avg MSE: 0.0215
Epoch 260 | D Loss: 106.4538 | Avg G Loss: 3.4297 | Avg MSE: 0.0216
Epoch 261 | D Loss: 106.4544 | Avg G Loss: 3.4297 | Avg MSE: 0.0214
Epoch 262 | D Loss: 106.4550 | Avg G Loss: 3.429

Epoch 372 | D Loss: 106.4535 | Avg G Loss: 3.4300 | Avg MSE: 0.0209
Epoch 373 | D Loss: 106.4536 | Avg G Loss: 3.4300 | Avg MSE: 0.0216
Epoch 374 | D Loss: 106.4537 | Avg G Loss: 3.4300 | Avg MSE: 0.0211
Epoch 375 | D Loss: 106.4530 | Avg G Loss: 3.4301 | Avg MSE: 0.0217
Epoch 376 | D Loss: 106.4534 | Avg G Loss: 3.4301 | Avg MSE: 0.0211
Epoch 377 | D Loss: 106.4536 | Avg G Loss: 3.4300 | Avg MSE: 0.0211
Epoch 378 | D Loss: 106.4529 | Avg G Loss: 3.4301 | Avg MSE: 0.0216
Epoch 379 | D Loss: 106.4533 | Avg G Loss: 3.4301 | Avg MSE: 0.0216
Epoch 380 | D Loss: 106.4535 | Avg G Loss: 3.4301 | Avg MSE: 0.0215
Epoch 381 | D Loss: 106.4530 | Avg G Loss: 3.4301 | Avg MSE: 0.0212
Epoch 382 | D Loss: 106.4534 | Avg G Loss: 3.4301 | Avg MSE: 0.0208
Epoch 383 | D Loss: 106.4535 | Avg G Loss: 3.4301 | Avg MSE: 0.0208
Epoch 384 | D Loss: 106.4531 | Avg G Loss: 3.4301 | Avg MSE: 0.0209
Epoch 385 | D Loss: 106.4527 | Avg G Loss: 3.4301 | Avg MSE: 0.0209
Epoch 386 | D Loss: 106.4528 | Avg G Loss: 3.430

Epoch 493 | D Loss: 106.4531 | Avg G Loss: 3.4308 | Avg MSE: 0.0212
Epoch 494 | D Loss: 106.4535 | Avg G Loss: 3.4308 | Avg MSE: 0.0209
Epoch 495 | D Loss: 106.4528 | Avg G Loss: 3.4308 | Avg MSE: 0.0213
Epoch 496 | D Loss: 106.4527 | Avg G Loss: 3.4308 | Avg MSE: 0.0212
Epoch 497 | D Loss: 106.4529 | Avg G Loss: 3.4308 | Avg MSE: 0.0213
Epoch 498 | D Loss: 106.4528 | Avg G Loss: 3.4308 | Avg MSE: 0.0211
Epoch 499 | D Loss: 106.4525 | Avg G Loss: 3.4308 | Avg MSE: 0.0210
Epoch 500 | D Loss: 106.4525 | Avg G Loss: 3.4308 | Avg MSE: 0.0214


In [5]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
from tqdm import tqdm

# Load fault texts
df = pd.read_excel("fault-logs-o-ran-350.xls")
#df = pd.read_excel("proj-training-12k.xlsx", engine="openpyxl")
fault_texts = df["Fault Description"].astype(str).tolist()

# Load MiniLM tokenizer + model
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

max_length = 128
token_level_embeddings = []

with torch.no_grad():
    for sentence in tqdm(fault_texts):
        encoded = tokenizer(sentence, padding="max_length", truncation=True,
                            max_length=max_length, return_tensors="pt").to(device)
        
        output = model(**encoded)  # output.last_hidden_state: [1, 128, 384]
        token_embeds = output.last_hidden_state.squeeze(0).cpu()  # [128, 384]
        
        token_level_embeddings.append(token_embeds)

# Save as a tensor [N, 128, 384]
token_level_embeddings = torch.stack(token_level_embeddings)
torch.save(token_level_embeddings, "minilm_token_embeddings.pt")



100%|████████████████████████████████████████| 350/350 [00:00<00:00, 359.15it/s]


In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Load fault logs
df = pd.read_excel("fault-logs-o-ran-350.xls")
#df = pd.read_excel("proj-training-12k.xlsx", engine="openpyxl")
fault_texts = df["Fault Description"].astype(str).tolist()

# Tokenizer and GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Dataset
class FaultDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.examples = tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    def __len__(self):
        return len(self.examples["input_ids"])
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.examples.items()}

dataset = FaultDataset(fault_texts, tokenizer)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned-fault",
    per_device_train_batch_size=4,
    num_train_epochs=10,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()
model.save_pretrained("gpt2-finetuned-fault")
tokenizer.save_pretrained("gpt2-finetuned-fault")


/tmp/ipykernel_4393/3540866464.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,4.126100
20,3.525800
30,3.064900
40,2.704600
50,2.581200
60,2.648700
70,2.564800
80,2.470800
90,2.024100
100,1.882500


('gpt2-finetuned-fault/tokenizer_config.json',
 'gpt2-finetuned-fault/special_tokens_map.json',
 'gpt2-finetuned-fault/vocab.json',
 'gpt2-finetuned-fault/merges.txt',
 'gpt2-finetuned-fault/added_tokens.json')

In [16]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load MiniLM token embeddings
minilm_token_embeds = torch.load("minilm_token_embeddings.pt")

# Load texts
df = pd.read_excel("fault-logs-o-ran-350.xls")
fault_texts = df["Fault Description"].astype(str).tolist()

# Load GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-finetuned-fault")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2-finetuned-fault").to(device)
gpt2.eval()

for p in gpt2.parameters():
    p.requires_grad = False

MAX_LEN = 128

class ProjectorDataset(Dataset):

    def __init__(self, embeds, texts):

        tok = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        self.embeds = embeds
        self.input_ids = tok["input_ids"]
        self.attn_mask = tok["attention_mask"]

        # ignore pad tokens
        self.labels = self.input_ids.clone()
        self.labels[self.attn_mask == 0] = -100

    def __len__(self):
        return len(self.embeds)

    def __getitem__(self, idx):
        return {
            "embed": self.embeds[idx],
            "labels": self.labels[idx],
            "mask": self.attn_mask[idx]
        }

dataset = ProjectorDataset(minilm_token_embeds, fault_texts)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

# projector
class EmbeddingProjector(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(384,512),
            nn.GELU(),
            nn.Linear(512,768),
            nn.GELU(),
            nn.Linear(768,768)
        )

    def forward(self,x):

        B,T,D = x.size()

        x = x.view(B*T,D)
        x = self.net(x)
        x = x.view(B,T,768)

        return x


projector = EmbeddingProjector().to(device)

optimizer = torch.optim.Adam(projector.parameters(), lr=1e-4)

EPOCHS = 300

for epoch in range(EPOCHS):

    projector.train()
    total_loss = 0

    for batch in loader:

        embeds = batch["embed"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        proj = projector(embeds)

        outputs = gpt2(
            inputs_embeds=proj,
            attention_mask=mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss {total_loss/len(loader):.4f}")

torch.save(projector.state_dict(),"trained_projector-new.pt")

Epoch 1/300 | Loss 11.6195
Epoch 2/300 | Loss 7.0127
Epoch 3/300 | Loss 6.5585
Epoch 4/300 | Loss 6.3562
Epoch 5/300 | Loss 6.1865
Epoch 6/300 | Loss 6.0581
Epoch 7/300 | Loss 5.9535
Epoch 8/300 | Loss 5.8589
Epoch 9/300 | Loss 5.7775
Epoch 10/300 | Loss 5.7050
Epoch 11/300 | Loss 5.6322
Epoch 12/300 | Loss 5.5842
Epoch 13/300 | Loss 5.5293
Epoch 14/300 | Loss 5.4641
Epoch 15/300 | Loss 5.4042
Epoch 16/300 | Loss 5.3391
Epoch 17/300 | Loss 5.2981
Epoch 18/300 | Loss 5.2631
Epoch 19/300 | Loss 5.2359
Epoch 20/300 | Loss 5.1901
Epoch 21/300 | Loss 5.1461
Epoch 22/300 | Loss 5.0886
Epoch 23/300 | Loss 5.0740
Epoch 24/300 | Loss 5.0076
Epoch 25/300 | Loss 4.9776
Epoch 26/300 | Loss 4.9498
Epoch 27/300 | Loss 4.8807
Epoch 28/300 | Loss 4.8717
Epoch 29/300 | Loss 4.8677
Epoch 30/300 | Loss 4.8259
Epoch 31/300 | Loss 4.7707
Epoch 32/300 | Loss 4.7544
Epoch 33/300 | Loss 4.7055
Epoch 34/300 | Loss 4.6856
Epoch 35/300 | Loss 4.6993
Epoch 36/300 | Loss 4.6475
Epoch 37/300 | Loss 4.6005
Epoch 38/

Epoch 298/300 | Loss 1.9368
Epoch 299/300 | Loss 1.9002
Epoch 300/300 | Loss 1.9022


In [21]:
import torch
import csv
from torch import nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Generator architecture
# -----------------------------
class Generator(nn.Module):

    def __init__(self, latent_dim=100, output_dim=384):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_dim,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,output_dim)
        )

    def forward(self,z):
        return self.model(z)


# -----------------------------
# Load trained generators
# -----------------------------
K = 30
generators = []

for j in range(1,K+1):

    G = Generator().to(device)

    G.load_state_dict(
        torch.load(f"trained_generator_{j}.pt",map_location=device)
    )

    G.eval()
    generators.append(G)

print(f"Loaded {K} generators")


# -----------------------------
# Load GPT-2
# -----------------------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-finetuned-fault")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2-finetuned-fault")
gpt2 = gpt2.to(device)
gpt2.eval()

print("GPT-2 loaded")


# -----------------------------
# Projector architecture
# (must match training)
# -----------------------------
class EmbeddingProjector(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(384,512),
            nn.GELU(),
            nn.Linear(512,768),
            nn.GELU(),
            nn.Linear(768,768)
        )

    def forward(self,x):

        B,T,D = x.size()

        x = x.view(B*T,D)
        x = self.net(x)
        x = x.view(B,T,768)

        return x


projector = EmbeddingProjector().to(device)

projector.load_state_dict(
    torch.load("trained_projector-new.pt",map_location=device)
)

projector.eval()

print("Projector loaded")


# -----------------------------
# Generation settings
# -----------------------------
latent_dim = 100
num_samples_per_gen = 500

prefix_len = 32

decoded_outputs = []

total_expected = K * num_samples_per_gen
sample_counter = 0

print(f"\nExpected samples: {total_expected}\n")


# -----------------------------
# Generate samples
# -----------------------------
with torch.no_grad():

    for g_idx,G in enumerate(generators):

        print(f"\n🔁 Generator {g_idx+1}/{K}")

        z = torch.randn(num_samples_per_gen,latent_dim).to(device)

        fake_emb = G(z)             # [N,384]

        # Convert sentence embedding → token embeddings
        fake_emb = fake_emb.unsqueeze(1).repeat(1,prefix_len,1)

        # Add small noise
        fake_emb = fake_emb + 0.01 * torch.randn_like(fake_emb)

        projected = projector(fake_emb)  # [N,prefix_len,768]

        for i in range(num_samples_per_gen):

            prefix = projected[i].unsqueeze(0)  # [1,prefix_len,768]

            output_ids = gpt2.generate(

                inputs_embeds=prefix,

                max_new_tokens=80,

                do_sample=True,

                temperature=0.8,

                top_k=40,

                top_p=0.9,

                repetition_penalty=1.15,

                pad_token_id=tokenizer.eos_token_id
            )

            decoded = tokenizer.decode(
                output_ids[0],
                skip_special_tokens=True
            )

            decoded_outputs.append(
                (f"G{g_idx+1}",f"Sample {i+1}",decoded)
            )

            sample_counter += 1

            # Show progress every 100 samples
            if sample_counter % 100 == 0:
                print(f"Progress: {sample_counter} / {total_expected}")


# -----------------------------
# Remove empty generations
# -----------------------------
decoded_outputs = [
    row for row in decoded_outputs
    if len(row[2].strip()) > 5
]


# -----------------------------
# Save results
# -----------------------------
output_file = "synthetic_fault_logs_final-2026-15k.csv"

with open(output_file,"w",newline="",encoding="utf-8") as f:

    writer = csv.writer(f)

    writer.writerow([
        "Generator",
        "Sample Number",
        "Generated Fault Description"
    ])

    for row in decoded_outputs:
        writer.writerow(row)


print(f"\nSaved {len(decoded_outputs)} samples to {output_file}")

Loaded 30 generators
GPT-2 loaded
Projector loaded

Expected samples: 15000


🔁 Generator 1/30
Progress: 100 / 15000
Progress: 200 / 15000
Progress: 300 / 15000
Progress: 400 / 15000
Progress: 500 / 15000

🔁 Generator 2/30
Progress: 600 / 15000
Progress: 700 / 15000
Progress: 800 / 15000
Progress: 900 / 15000
Progress: 1000 / 15000

🔁 Generator 3/30
Progress: 1100 / 15000
Progress: 1200 / 15000
Progress: 1300 / 15000
Progress: 1400 / 15000
Progress: 1500 / 15000

🔁 Generator 4/30
Progress: 1600 / 15000
Progress: 1700 / 15000
Progress: 1800 / 15000
Progress: 1900 / 15000
Progress: 2000 / 15000

🔁 Generator 5/30
Progress: 2100 / 15000
Progress: 2200 / 15000
Progress: 2300 / 15000
Progress: 2400 / 15000
Progress: 2500 / 15000

🔁 Generator 6/30
Progress: 2600 / 15000
Progress: 2700 / 15000
Progress: 2800 / 15000
Progress: 2900 / 15000
Progress: 3000 / 15000

🔁 Generator 7/30
Progress: 3100 / 15000
Progress: 3200 / 15000
Progress: 3300 / 15000
Progress: 3400 / 15000
Progress: 3500 / 15000



In [44]:
# fine_tune_gpt2.py
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Load fault logs
df = pd.read_excel("fault-logs-o-ran-12k.xlsx", engine="openpyxl")
fault_texts = df["Fault Description"].astype(str).tolist()

# Tokenizer and GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Dataset
class FaultDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.examples = tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    def __len__(self):
        return len(self.examples["input_ids"])
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.examples.items()}

dataset = FaultDataset(fault_texts, tokenizer)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned-fault-12k",
    per_device_train_batch_size=4,
    num_train_epochs=8,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()
model.save_pretrained("gpt2-finetuned-fault-12k")
tokenizer.save_pretrained("gpt2-finetuned-fault-12k")


/tmp/ipykernel_544054/2898927493.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,3.233500
20,2.624800
30,2.365300
40,2.075200
50,2.047300
60,1.975300
70,2.254500
80,2.024500
90,1.928500
100,1.733600


('gpt2-finetuned-fault-12k/tokenizer_config.json',
 'gpt2-finetuned-fault-12k/special_tokens_map.json',
 'gpt2-finetuned-fault-12k/vocab.json',
 'gpt2-finetuned-fault-12k/merges.txt',
 'gpt2-finetuned-fault-12k/added_tokens.json')

In [11]:
# generate_faults_from_gan.py
import torch
import csv
from torch import nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# --- Generator class ---
class Generator(nn.Module):
    def __init__(self, latent_dim=100, output_dim=384):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
    def forward(self, z):
        return self.model(z)

# --- Projector class (must match training structure) ---
class EmbeddingProjector(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(384, 768)
    def forward(self, x):
        return self.linear(x)

# Config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_dim = 100
K = 20
num_samples_per_gen = 60

# --- Load all trained generators ---
generators = []
for j in range(1, K + 1):
    G = Generator().to(device)
    G.load_state_dict(torch.load(f"trained_generator_{j}.pt", map_location=device))
    G.eval()
    generators.append(G)

# --- Load fine-tuned GPT-2 ---
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-finetuned-fault")
tokenizer.pad_token = tokenizer.eos_token
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2-finetuned-fault").to(device)
gpt2.eval()

# --- Load trained projector ---
projector = EmbeddingProjector().to(device)
projector.load_state_dict(torch.load("trained_projector-old.pt", map_location=device))
projector.eval()

# --- Generate and decode ---
decoded_outputs = []

with torch.no_grad():
    for g_idx, G in enumerate(generators):
        print(f"Generating from Generator {g_idx + 1}...")
        z = torch.randn(num_samples_per_gen, latent_dim).to(device)
        fake_emb = G(z)                          # [N, 384]
        projected = projector(fake_emb)          # [N, 768]

        for i, emb in enumerate(projected):
            input_emb = emb.unsqueeze(0).unsqueeze(1)  # [1, 1, 768]
            output_ids = gpt2.generate(
                inputs_embeds=input_emb,
                max_length=128,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
            decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            decoded_outputs.append((f"G{g_idx + 1}", f"Sample {i + 1}", decoded))
            print(f"✓ G{g_idx + 1}, Sample {i + 1}: {decoded[:60]}...")

# --- Save to CSV ---
output_file = "inference-data-new.csv"
with open(output_file, mode="w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Generator", "Sample Number", "Generated Fault Description"])
    for row in decoded_outputs:
        writer.writerow(row)

print(f"\n✅ Saved {len(decoded_outputs)} samples to {output_file}")


/tmp/ipykernel_4393/1602663025.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  G.load_state_dict(torch.load(f"trained_generator_{j}.pt", map_location=device))
/tmp/ipyk

Generating from Generator 1...
✓ G1, Sample 1:  is not available. [ 0 ] could not assign _ rrc _ nr _ idx _...
✓ G1, Sample 2: .o.d.u.i.c.RequestInterfaceSetup() /home/oran/openairinterfa...
✓ G1, Sample 3:  ( 1 ; 31m [ nr _ rrc ] [ dl ] ( cellid bc614e , ue id 1 rnt...
✓ G1, Sample 4: .exe /home / oran / openairinterface5g / openair2 / pdrd / ‘...
✓ G1, Sample 5: .exe /home/oran/openairinterface5g/openair2/common/pfcp-sm.c...
✓ G1, Sample 6:  the server was not available for reading/writing. (In this ...
✓ G1, Sample 7:  ausdallas-nas-amfd[13098]: 02/13/2015 sgcpcd[12882]: 02/13/...
✓ G1, Sample 8:  ( ) failed! in rx _ rx _ nr _ prach _ ru ( ) / home / oran ...
✓ G1, Sample 9:  the table [ 0m # ‘ 2a31m ‘ hw ] received unsuccessful resul...
✓ G1, Sample 10: /nrf - oai - nf_config - ue 1 [ 2015 - 11 18 : 14 : 14 ] [ a...
✓ G1, Sample 11:  [ 0m [ nas ] sctp client connects[ 0 ] ( ) to ‘ ‘ ‘ 10. 0. ...
✓ G1, Sample 12:  [ 0m # [ 93m [ mac ] ue 2aa … : request release after ul fa...
✓ G1, 

✓ G2, Sample 42: .net.UnknownHostException: failed to initialize Fd [0] (../s...
✓ G2, Sample 43:  the sib1.conf file. …. ….. /.. /etc/rpi/conf. …. ….. /etc/r...
✓ G2, Sample 44:  at the time of registration. (3) Subscriber provided a curr...
✓ G2, Sample 45: , 93rd Session - NRG-HCTS-NR-01 Session# [2028-03-11 14:35:1...
✓ G2, Sample 46:  - 1 [ 1 ; 31m [ rrc ] unknown ciphering algorithm ” e2aa7 #...
✓ G2, Sample 47:  - 1 [ pool - 1 ] [ peer ] file.. /.. /.. / targets / projec...
✓ G2, Sample 48: [ 0m # ” 0m # ” 0m # [ 93m [ phy ] connect() to ‘ ‘ 0m ‘ ” f...
✓ G2, Sample 49:  to connect to the server (eg, 127.0.0.1:2543)Feb 17 22:21:0...
✓ G2, Sample 50:  to connect to the server. o. o. d. e. h. s. s. s. s. v. e. ...
✓ G2, Sample 51:  of 6.0.1 [drdw] Remove "libdrd[0].plmn" line 93. Remove "-o...
✓ G2, Sample 52:  that ran at a time of 10:30am PST for the current session.P...
✓ G2, Sample 53: : 02 / 13 / 11 : 44 : 59. 106 : [ sbi ] warning : [ 14100714...
✓ G2, Sample 54:  this secti

✓ G4, Sample 23:  [9340] [amf_app] [debug] Initializing SBIX SEQ (../lib/sbi/...
✓ G4, Sample 24:  28 [displayText] [SIC] [com.google.gson.jsonobjective.inter...
✓ G4, Sample 25:  - 12:54:35 srs-ran open5gs-udrd[14925]: 02/13/12 13:54:35.6...
✓ G4, Sample 26:  this section ( ‘ ‘ ” ) and ‘ ‘ ” ” ” ” ” ” ” ” ” ” ” ” ” ” ...
✓ G4, Sample 27:  10 23:54:53 srs-ran open5gs-scpd[15095]: 02/10 23:54:53.436...
✓ G4, Sample 28:  the config file : /opt/app-config.yaml file.. /.. /.. / tar...
✓ G4, Sample 29: , 12 May 2018 19:21:46 srs-ran open5gs-nbcs-seppd[14930]: [s...
✓ G4, Sample 30:  ( nr - numelt > 0 && nr - > max ( 1 / nr - numelt * sizeof ...
✓ G4, Sample 31:  this section is not a section of the app.it is a configurat...
✓ G4, Sample 32:  1 h 1 h - 2 nr - 1 ; s ssb - app ] info : [ 142a196a - ea30...
✓ G4, Sample 33:  ( 1 / 3 ) failed! in rx _ prach _ ru ( ) / home / oran / ‘ ...
✓ G4, Sample 34:  the numerals with a periodicity of 0.05 (or, the numerals w...
✓ G4, Sample 35:  [ 0m # ‘ 1

✓ G6, Sample 4: -NR-3f-sgfr-amfd[13900]: [sbi] WARNING: [7] Failed to assign...
✓ G6, Sample 5:  1 [ gtpu ] unknown format d [ hw ] initiating sync transfer...
✓ G6, Sample 6:  the system was unsuccessful. Try again later. No data avail...
✓ G6, Sample 7:  to get the client to start up.I ran the configuration file ...
✓ G6, Sample 8: 1 session##[GCC] [debug] A gnb request was received f...
✓ G6, Sample 9:  and GPRS (256), and FPU (256), the SCTP module is running.
...
✓ G6, Sample 10: , 1 ( ‘ 0 ‘ ‘ ‘ ‘ 0 ) failed! in encode_numerology ( ) / hom...
✓ G6, Sample 11: 
 ( nr - > 0 ) failed! in clone _ rrc _ nr _ conﬁg ‘ config ...
✓ G6, Sample 12:  this time. Give the band the time to reactivate and start t...
✓ G6, Sample 13:  and ‘ ‘ – provided a result with no value, informing the cl...
✓ G6, Sample 14: .exe / og - ldrd / uesoft ‘ ‘ vai ‘ - pdss ”common / uesoft ...
✓ G6, Sample 15:  to read config_list_size() /home/oran/openairinterface5g/op...
✓ G6, Sample 16:  is 0xe000, ” hw ” is th

...

✓ G12, Sample 46:  - 1 - gpu-scm [ 12976 ] : [ sbi ] warning : retry registrat...
✓ G12, Sample 47:  the band has provided.I ran it out of the box, i.e. 1 mac, ...
✓ G12, Sample 48:  all config values have the format json encoded.I added the ...
✓ G12, Sample 49:  13 23:59:58 srs-ran open5gs-nbclbs2gs-smfd[14900]: 02/13 23...
✓ G12, Sample 50:  ( gnb _ app _ pf _ app ) failed! in clone_pf_configcommon (...
✓ G12, Sample 51: .targets[NR_APP] is not available#[NGAP] no amf_sg_co...
✓ G12, Sample 52: .targets[NR_APP] /home/oran/openairinterface5g/openair2/RRC/...
✓ G12, Sample 53:  the client sends out an HTTP request. Parameters: band : in...
✓ G12, Sample 54:  – 12 – 17 : 44 : 59 srs - ran open5gs - pcfd [ 12999 ] : 02...
✓ G12, Sample 55:  a rnti thread ngInitiated : sw error : ent : 255 inst : 255...
✓ G12, Sample 56: . o. d. s. s. s. c. f. s. e. c. i. s. s. p. a. d. s. s. c. h...
✓ G12, Sample 57:  1 time out of 2,000,000 in the app. ” 134493 failed, errno ...
✓ G12, Sample 58:  amf 

✓ G14, Sample 26:  and 10 min ago srs-ran open5gs-smf-drd[13099]: [sbi] WARNIN...
✓ G14, Sample 27: .exe /home/oran/openairinterface5g/openair2/RRC/NR/nr_rrc_co...
✓ G14, Sample 28:  - 1 - 8 ) failed! in clone _ pf _ configcommon ( ) / home /...
✓ G14, Sample 29:  - 03 - 18 19 : 00 : 53. 256 ] [ ngap ] [ debug ] remove ng ...
✓ G14, Sample 30: .app.xpi.xpiProviderNullPeriodicUpdateEvent |.. /.. / lib / ...
✓ G14, Sample 31:  ( 0m ‘ 1 ; 31m ” 1 rrc ‘ 0m ” c : cmines [ hw ] request dat...
✓ G14, Sample 32:  at the start of the current session. ( ‘ ‘ ‘ ‘ ‘ ‘ failed ‘...
✓ G14, Sample 33:  and 12.04 – 03 – 21 : 08. 800 ” out of order ” ‘ noaa ‘ pd ...
✓ G14, Sample 34:  nr - threadcreate ( ) : error - / o - du - l2 / src / cm / ...
✓ G14, Sample 35: : 0031PM PDT / 12 May – 03:30PM EDT / 619:30 IST / 9528Km [ ...
✓ G14, Sample 36: . sgcp - ue 1 [.. / src / cm / cm _ inet. c : 169 ] file.. /...
✓ G14, Sample 37:  ( 1 / nr - regen ) failed! in rrc _ rrc _ pd [ 0mcmdline : ...
✓ G14, Sample 38

✓ G16, Sample 6: , 21 Sep 18 18 21:54:22 srs-ran open5gs-smfd[14976]: 02/19 1...
✓ G16, Sample 7: , 12 August 2015: 21:59:54 srs-ran open5gs-seppd[13098]: 02/...
✓ G16, Sample 8:  that isn't the end of it. Try to connect to the server and ...
✓ G16, Sample 9:  [ 0m # [ 93m [ mac ] ue 0xe000 [ 2a02 # [ 0m [ rrc ] receiv...
✓ G16, Sample 10:  ( 1 - > 0 && nr _ rrc _ rrc _ rnti > 0 && nr _ rrc _ nr _ r...
✓ G16, Sample 11:  the length of the buffer: 8] (NB8) Node error: SubSIDR f...
✓ G16, Sample 12: 
 ( 1 / 4 in ‘ 1, 2 ‘ ‘ ‘ ‘ ‘ 0 ) failed! in rx _ rx _ nr _ ...
✓ G16, Sample 13:  at 14:44:25 srs-ran open5gs-udrd[14963]: 02/12/14 21:44:25....
✓ G16, Sample 14:  12 22 : 21 : 21 : 07 srs - ran open5gs - seppd [ 14940 ] : ...
✓ G16, Sample 15:  1 – 5 of 7 – Oct 25 22:54:59 srs-ran open5gs-bsfd[14928]: 0...
✓ G16, Sample 16:  – 0m # ‘ ‘ srs - ran open5gs - scpd [ 14976 ] : [ sbi ] war...
✓ G16, Sample 17:  the buffer is not properly initialized, errno ( 106 ) == - ...
✓ G16, Sample 18: . c :

✓ G17, Sample 46:  12:35:54 srs-ran open5gs-amfd[14922]: 02/12/14 19:04:03.256...
✓ G17, Sample 47: .srs-ran open5gs-cmogs-rlc-xrlc2fs[13691]: 02/13/12 16:59:25...
✓ G17, Sample 48:  with a buffer of 256 bytes. Subsequent buffer initializatio...
✓ G17, Sample 49:  28 23 : 44 : 59 srs - ran open5gs - udrd [ 14976 ] : 02 / 1...
✓ G17, Sample 50:  at the time of registration and received it. (Februy 23rd, ...
✓ G17, Sample 51:  and ‘ gtpu_idx : 16 ciphering priority : 2 ciphering algori...
✓ G17, Sample 52:  (posix ) - > max ( 4 * ( table - > idx * 2 - > max ( nr _ t...
✓ G17, Sample 53:  is not available to all gnb 0. 1. 1 is not available. 2 is ...
✓ G17, Sample 54:  the numerator and denominator values in the cell. Parameter...
✓ G17, Sample 55:  to get the rrc_nr_rrc_nr and rrc_nr_nr_bandwidth values. 0m...
✓ G17, Sample 56:  a file imd[..rp] o.o.d.b.c.t.o.u.f.s.r.c.o.i.d.c.x.e.p.x.rd...
✓ G17, Sample 57: .scm: error while encoding file[-1][amfd[HW] Trying to conne...
✓ G17, Sample 58

✓ G19, Sample 26: , ‘ 1 : ‘ ‘ down, ‘ ‘ ‘ ‘ ‘ ‘ ‘ ‘ ‘ ‘ down ‘ ‘ ‘ ‘ ‘ down ‘ ...
✓ G19, Sample 27:  for the current session.ERROR! Failed to initialize PDO (.....
✓ G19, Sample 28: -dru-rs-3g-du-smf-sc-cmfs' centralized directory /etc/rfc14s...
✓ G19, Sample 29:  is not standardized in the syntax, or in the syntaxes for t...
✓ G19, Sample 30:  this section.CMDLINE: dl_nr_esc_intval:cell_id 1378 cells: ...
✓ G19, Sample 31: -ran : 0030 : 44 srs - ran open5gs - bsfd [ 12976 ] : 02 / 1...
✓ G19, Sample 32:  - 12 21 : 21 : 25 srs - ran open5gs - bsfd [ 14940 ] : 02 /...
✓ G19, Sample 33: , nr _ rrc _ prach _ prach _ ruised _ ue _ imd _ imd _ in _ ...
✓ G19, Sample 34: . o. o. d. c. s. i. e. f. s. r. e. g. u. pld. ue. ue. pld. u...
✓ G19, Sample 35:  the problem! ‘ ‘ ” 0 ‘ ‘ ‘ ” ( 1 ‘ ‘ ‘ ‘ 1 ) is not the val...
✓ G19, Sample 36:  2 o'clock. ‘ sctp : ‘ … sctp shutdown ” ” 0x1 parameters : ...
✓ G19, Sample 37:  [ 0m # ” 00m [ pfcp ] connect : ” failed ” port 7777 # ” 00...
✓ G19, Sample 38

In [5]:
import pandas as pd

# Group outputs by generator index
gen_ids = []
texts = []

for g_idx in range(K):
    for i in range(num_samples_per_gen):
        idx = g_idx * num_samples_per_gen + i
        gen_ids.append(f"G{g_idx + 1}")
        texts.append(decoded_outputs[idx])

# Create DataFrame with generator ID
df = pd.DataFrame({
    "Generator": gen_ids,
    "Synthetic Fault Log": texts
})

# Save to CSV
df.to_csv("synthetic_fault_logs_by_generator-12k.csv", index=False)
print("✅ Saved to synthetic_fault_logs_by_generator-12k.csv")



✅ Saved to synthetic_fault_logs_by_generator.csv


In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge

# Load data with generator column
df_fake = pd.read_csv("gmm_inference_evaluation.csv")
fake_logs = df_fake["Synthetic Fault Log"].dropna().astype(str).tolist()
generator_ids = df_fake["Generator"].tolist()

# Load real logs
df_real = pd.read_excel("fault-logs-o-ran-350.xls")
real_logs = df_real["Fault Description"].dropna().astype(str).tolist()

# Models
embedder = SentenceTransformer("all-MiniLM-L6-v2")
rouge = Rouge()
smoothie = SmoothingFunction().method1

# Evaluation records
records = []

for i, fake in enumerate(fake_logs):
    fake_emb = embedder.encode([fake])
    real_embs = embedder.encode(real_logs)
    sims = cosine_similarity(fake_emb, real_embs)[0]
    best_idx = sims.argmax()
    ref = real_logs[best_idx]

    # BLEU & ROUGE
    bleu = sentence_bleu([ref.split()], fake.split(), smoothing_function=smoothie)
    rouge_score = rouge.get_scores(fake, ref)[0]['rouge-l']['f']
    cosine_sim = sims[best_idx]

    records.append({
        "Generator": generator_ids[i],
        "Fake Log": fake,
        "Best Match Real Log": ref,
        "BLEU": bleu,
        "ROUGE_L_F1": rouge_score,
        "Cosine Similarity": cosine_sim
    })

# Save detailed evaluation
results_df = pd.DataFrame(records)
results_df.to_csv("synthetic_fault_log_eval_detailed.csv", index=False)

# Summary metrics per generator
summary_df = results_df.groupby("Generator")[["BLEU", "ROUGE_L_F1", "Cosine Similarity"]].mean()
summary_df.reset_index(inplace=True)
summary_df.to_csv("evaluation_metrics_by_generator.csv", index=False)

print("✅ Evaluation complete. Metrics saved to:")
print("- synthetic_fault_log_eval_detailed.csv")
print("- evaluation_metrics_by_generator.csv")


KeyError: 'Generator'